# Exploratory Data Analysis: Credit Risk Probability Model

This notebook explores the Xente transaction dataset for the Bati Bank credit risk challenge. It is intentionally exploratory; production transformations live in `src/data_processing.py`.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = Path('../data/raw/training.csv')
if not DATA_PATH.exists():
    candidates = list(Path('../data/raw').glob('*.csv'))
    if candidates:
        DATA_PATH = candidates[0]
    else:
        raise FileNotFoundError('Place the Xente CSV in data/raw/, for example data/raw/training.csv')

df = pd.read_csv(DATA_PATH)
df.head()

FileNotFoundError: Place the Xente CSV in data/raw/, for example data/raw/training.csv

## 1. Overview of the data

In [ ]:
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]:,}')
df.info()

In [ ]:
df.head()

## 2. Summary statistics

In [ ]:
df.describe(include='all').T

## 3. Missing values

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_percent = (missing / len(df) * 100).round(2)
missing_table = pd.DataFrame({'missing_count': missing, 'missing_percent': missing_percent})
missing_table[missing_table['missing_count'] > 0]

## 4. Numerical feature distributions

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
for col in numeric_cols:
    plt.figure(figsize=(7, 4))
    df[col].dropna().hist(bins=50)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.savefig(f'../reports/figures/hist_{col}.png')
    plt.show()

## 5. Categorical feature distributions

In [ ]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
ignore_high_cardinality = {'TransactionId', 'BatchId', 'AccountId', 'SubscriptionId', 'CustomerId'}
for col in [c for c in categorical_cols if c not in ignore_high_cardinality]:
    counts = df[col].value_counts().head(15)
    plt.figure(figsize=(8, 4))
    counts.plot(kind='bar')
    plt.title(f'Top categories for {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(f'../reports/figures/bar_{col}.png')
    plt.show()

## 6. Correlation analysis

In [ ]:
if len(numeric_cols) > 1:
    corr = df[numeric_cols].corr()
    plt.figure(figsize=(8, 6))
    plt.imshow(corr, aspect='auto')
    plt.colorbar()
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
    plt.yticks(range(len(corr.index)), corr.index)
    plt.title('Correlation heatmap')
    plt.tight_layout()
    plt.savefig('../reports/figures/correlation_heatmap.png')
    plt.show()
    display(corr)
else:
    print('Not enough numerical columns for correlation analysis.')

## 7. Outlier detection with box plots

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(7, 3))
    plt.boxplot(df[col].dropna(), vert=False)
    plt.title(f'Box plot of {col}')
    plt.xlabel(col)
    plt.savefig(f'../reports/figures/box_{col}.png')
    plt.show()

## 8. Customer-level RFM exploration

In [ ]:
required = {'CustomerId', 'TransactionStartTime', 'Amount', 'Value'}
if required.issubset(df.columns):
    work = df.copy()
    work['TransactionStartTime'] = pd.to_datetime(work['TransactionStartTime'], errors='coerce', utc=True)
    snapshot_date = work['TransactionStartTime'].max() + pd.Timedelta(days=1)
    rfm = (
        work.groupby('CustomerId')
        .agg(
            last_transaction=('TransactionStartTime', 'max'),
            frequency=('TransactionId', 'count') if 'TransactionId' in work.columns else ('Amount', 'count'),
            monetary=('Value', 'sum'),
            total_amount=('Amount', 'sum'),
        )
        .reset_index()
    )
    rfm['recency'] = (snapshot_date - rfm['last_transaction']).dt.days
    display(rfm[['recency', 'frequency', 'monetary', 'total_amount']].describe())
    for col in ['recency', 'frequency', 'monetary']:
        plt.figure(figsize=(7, 4))
        rfm[col].hist(bins=50)
        plt.title(f'Customer-level {col}')
        plt.xlabel(col)
        plt.ylabel('Customers')
        plt.savefig(f'../reports/figures/rfm_{col}.png')
        plt.show()
else:
    print(f'Missing required columns for RFM: {required.difference(df.columns)}')

## 9. Top 3-5 EDA insights

Update this cell after running the notebook on the real dataset. Suggested structure:

1. **Transaction size and skewness:** Amount/Value distributions show [describe actual skew/outliers].
2. **Customer activity concentration:** A small/large group of customers contributes [describe actual transaction concentration].
3. **Categorical concentration:** ProductCategory/ChannelId are concentrated in [top categories].
4. **Data quality:** Missing values are [low/moderate/high] and mainly affect [columns].
5. **Feature engineering implication:** RFM aggregation is appropriate because [actual evidence from frequency, monetary, recency].